In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Muat data yang sudah dibersihkan
df = pd.read_csv("/content/cleaned_dataset.csv")

# Encoding Kategorikal (Target)
le = LabelEncoder()
y = le.fit_transform(df['Subkategori'])
num_classes = len(le.classes_)

# Tokenisasi dan Ekstraksi Matriks Teks
texts = df['Deskripsi Teks (Untuk Embeddings)'].astype(str).tolist()
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
max_length = max([len(x) for x in sequences])

# Standardisasi dimensi matriks (Padding)
X_padded = pad_sequences(sequences, maxlen=max_length, padding='post')
X_train, X_val, y_train, y_val = train_test_split(X_padded, y, test_size=0.2, random_state=42)

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input

# Komputasi Arsitektur Ekstraksi Fitur Lokal
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=5000, output_dim=32),
    # Conv1D berfungsi sebagai ekstraktor frasa (n-grams) untuk pola teks
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    # GlobalMaxPooling1D mereduksi dimensi dan menahan sinyal fitur terkuat
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3), # Proteksi Overfitting
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 96, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 94, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 171,538 (670.07 KB)

 Trainable params: 171,538 (670.07 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Pelatihan Model
history = model.fit(X_train, y_train, epochs=15, validation_data=(X_val, y_val), verbose=1)

# Simpan model untuk digunakan pada tahap produksi (src/models/intent_classifier.py)
model_path = "/content/intent_classification_model.keras"
model.save(model_path)
print(f"Model berhasil diekspor ke: {model_path}")

Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9975 - loss: 0.1009 - val_accuracy: 1.0000 - val_loss: 0.0524
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9950 - loss: 0.0771 - val_accuracy: 1.0000 - val_loss: 0.0392
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9975 - loss: 0.0667 - val_accuracy: 1.0000 - val_loss: 0.0310
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9975 - loss: 0.0561 - val_accuracy: 1.0000 - val_loss: 0.0248
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 1.0000 - loss: 0.0525 - val_accuracy: 1.0000 - val_loss: 0.0199
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 0.0398 - val_accuracy: 1.0000 - val_loss: 0.0162
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 1.0000 - loss: 0.0329 - val_accuracy: 1.0000 - val_loss: 0.0137
Epoch 8/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9975 - loss: 0.0339 - val_accuracy: 1.0000 - v